# Thắng

# Dự án: Echo Valley - Dự án Phân tích Thương mại điện tử Olist
**Tên Notebook:** 02_postgresql_pipeline.ipynb
**Mục tiêu:** Xây dựng pipeline đẩy dữ liệu từ CSV vào PostgreSQL và thực hiện JOIN 9 bảng để tạo ra Master Table (bảng tổng hợp) cho phân tích.


# 1. Kết nối cơ sở dữ liệu

## 1.1 Cấu hình kết nối

### 1.1.1 Khai báo thông tin xác thực

**a.** Sử dụng `python-dotenv` để load biến môi trường (không hard-code mật khẩu)

In [5]:
# Code import sqlalchemy, dotenv và thiết lập chuỗi kết nối (connection string)
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Nạp biến môi trường từ file .env
load_dotenv()

# Lấy chuỗi kết nối từ biến môi trường
DATABASE_URL = os.getenv("DATABASE_URL")

# Thiết lập kết nối
try:
    engine = create_engine(DATABASE_URL)
    connection = engine.connect()
    print("Kết nối PostgreSQL thành công!")
except Exception as e:
    print("Lỗi kết nối:", e)


Kết nối PostgreSQL thành công!


**Nhận xét:**
- [Điền nhận xét: Kết nối đã thành công hay chưa?]

# 2. Xây dựng Pipeline đẩy dữ liệu (ETL)

## 2.1 Tạo cấu trúc bảng (Schema)

### 2.1.1 Đẩy dữ liệu từ CSV lên Database

**a.** Vòng lặp đọc CSV và dùng `to_sql` để tạo bảng

In [6]:
# Code đọc 9 file CSV và push lên PostgreSQL
import time

data_dir = '../data/archive/'
files = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

start_time = time.time()
print("Bắt đầu đẩy dữ liệu lên PostgreSQL...")

for table_name, file_name in files.items():
    file_path = os.path.join(data_dir, file_name)
    df = pd.read_csv(file_path)
    
    # Push dataframe lên db
    print(f"Đang đẩy bảng {table_name} ({len(df)} dòng)...")
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    
print(f"Đẩy dữ liệu hoàn tất trong {time.time() - start_time:.2f} giây!")


Bắt đầu đẩy dữ liệu lên PostgreSQL...
Đang đẩy bảng customers (99441 dòng)...
Đang đẩy bảng geolocation (1000163 dòng)...
Đang đẩy bảng order_items (112650 dòng)...
Đang đẩy bảng order_payments (103886 dòng)...
Đang đẩy bảng order_reviews (99224 dòng)...
Đang đẩy bảng orders (99441 dòng)...
Đang đẩy bảng products (32951 dòng)...
Đang đẩy bảng sellers (3095 dòng)...
Đang đẩy bảng category_translation (71 dòng)...
Đẩy dữ liệu hoàn tất trong 58.49 giây!


**Nhận xét:**
- [Điền nhận xét: Dữ liệu đã đẩy thành công, có lỗi định dạng kiểu dữ liệu nào không?]

# 3. Thực hiện Join 9 bảng (Core Task)

## 3.1 Xây dựng câu truy vấn SQL

### 3.1.1 Viết truy vấn JOIN

**a.** Kết hợp các bảng chính (Orders, Items, Customers, Products, Payments, Reviews...)

In [7]:
# Code viết truy vấn SQL JOIN 9 bảng và lưu kết quả vào một dataframe/table mới
query = """
SELECT 
    o.order_id, o.customer_id, o.order_status, o.order_purchase_timestamp,
    c.customer_unique_id, c.customer_zip_code_prefix, c.customer_city, c.customer_state,
    oi.order_item_id, oi.product_id, oi.seller_id, oi.price, oi.freight_value,
    p.product_category_name, pt.product_category_name_english,
    s.seller_zip_code_prefix, s.seller_city, s.seller_state,
    pay.payment_type, pay.payment_installments, pay.payment_value,
    rev.review_score
FROM 
    orders o
JOIN 
    customers c ON o.customer_id = c.customer_id
JOIN 
    order_items oi ON o.order_id = oi.order_id
JOIN 
    products p ON oi.product_id = p.product_id
LEFT JOIN 
    category_translation pt ON p.product_category_name = pt.product_category_name
JOIN 
    sellers s ON oi.seller_id = s.seller_id
LEFT JOIN 
    order_payments pay ON o.order_id = pay.order_id
LEFT JOIN 
    order_reviews rev ON o.order_id = rev.order_id
"""

print("Đang thực hiện truy vấn JOIN...")
start_time = time.time()

# Đọc kết quả vào DataFrame
df_master = pd.read_sql_query(query, engine)

print(f"Truy vấn hoàn tất trong {time.time() - start_time:.2f} giây!")
print(f"Kích thước Master Table: {df_master.shape}")


Đang thực hiện truy vấn JOIN...
Truy vấn hoàn tất trong 5.22 giây!
Kích thước Master Table: (118310, 22)


**Nhận xét:**
- [Điền nhận xét: Truy vấn chạy mất bao lâu? Master Table có bao nhiêu dòng và cột?]

# 4. Kiểm tra Master Table

## 4.1 Kiểm tra dữ liệu tổng hợp

### 4.1.1 Kiểm tra null và kiểu dữ liệu của bảng cuối

**a.** Kiểm tra toàn vẹn dữ liệu

In [8]:
# Code kiểm tra df_master.info() và df_master.head()
print("--- Thông tin Master Table ---")
df_master.info()

print("\n--- 5 dòng đầu tiên ---")
display(df_master.head())


--- Thông tin Master Table ---
<class 'pandas.DataFrame'>
RangeIndex: 118310 entries, 0 to 118309
Data columns (total 22 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       118310 non-null  str    
 1   customer_id                    118310 non-null  str    
 2   order_status                   118310 non-null  str    
 3   order_purchase_timestamp       118310 non-null  str    
 4   customer_unique_id             118310 non-null  str    
 5   customer_zip_code_prefix       118310 non-null  int64  
 6   customer_city                  118310 non-null  str    
 7   customer_state                 118310 non-null  str    
 8   order_item_id                  118310 non-null  int64  
 9   product_id                     118310 non-null  str    
 10  seller_id                      118310 non-null  str    
 11  price                          118310 non-null  float64
 12  freight_va

,order_id,customer_id,order_status,order_purchase_timestamp,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,...,freight_value,product_category_name,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_type,payment_installments,payment_value,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,1,4244733e06e7ecb4970a6e2683c13e61,...,13.29,cool_stuff,cool_stuff,27277,volta redonda,SP,credit_card,2.0,72.19,5.0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,1,e5f2d52b802189ee658865ca93d83a8f,...,19.93,pet_shop,pet_shop,3471,sao paulo,SP,credit_card,3.0,259.83,4.0
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,1,c777355d18b72b67abbeef9df44fd0fd,...,17.87,moveis_decoracao,furniture_decor,37564,borda da mata,MG,credit_card,5.0,216.87,5.0
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,1,7634da152a4610f1595efa32f14722fc,...,12.79,perfumaria,perfumery,14403,franca,SP,credit_card,2.0,25.78,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,1,ac6c3623068f30de03045865e4e10089,...,18.14,ferramentas_jardim,garden_tools,87900,loanda,PR,credit_card,3.0,218.04,5.0


**Nhận xét:**
- [Điền nhận xét: Bảng cuối cùng đã sẵn sàng cho phân tích chưa?]